In [ ]:
# --- Paths (repo-relative; this notebook runs from notebooks/) ---
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA      = PROJECT_ROOT / 'data'
RAW       = DATA / 'raw'          # external source data (read-only)
EXTERNAL  = DATA / 'external'     # frozen third-party / HPC-derived inputs (read-only)
ANNOTATED = DATA / 'annotated'    # tables derived by these notebooks
FIGURES   = PROJECT_ROOT / 'figures'
# Gene ages come from GenOrigin via an ENSG->UniProt table that exists only on the AKEY cluster;
# off-cluster the per-gene ages it produced are read from data/external/hpc_snapshot/.


# Load Packages

In [ ]:
# Environment: `pip install -r requirements.txt` from the repository root (see README.md).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu, spearmanr, pearsonr, gaussian_kde, wilcoxon
from matplotlib.patches import Patch
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Master file: all human Swiss-Prot canonical + alternative isoforms with IDR annotations
all_isoforms_df = pd.read_csv("../data/annotated/IDRisoforms_df.csv")

# Master file: only TF proteins + their isoforms
tf_isoforms_df = pd.read_csv("../data/annotated/TranscriptionIsoforms_df.csv")

print("All isoforms:", all_isoforms_df.shape)
print("TF isoforms:", tf_isoforms_df.shape)

print("\nTop TF families:")
print(all_isoforms_df.loc[all_isoforms_df["is_tf"], "tf_family"].value_counts().head(10))

# Header Annotations Legend

| Column                            | Meaning                                                                                                                     |
| --------------------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| `db`                              | UniProt database source. `sp` = Swiss-Prot reviewed protein.                                                        |
| `Entry`                           | UniProt accession for this sequence. Can canonical/isoform-specific like `P04637` or `P04637-2`.              |
| `prot_info`                       | Protein name taken from FASTA header.                                                                                  |
| `gene_name`                       | Gene symbol/name parsed from UniProt/FASTA.                                                                                 |
| `prot_existence`                  | 1-5 UniProt protein existence evidence level from FASTA. Lower = stronger evidence.                                    |
| `prot_seqVersion`                 | UniProt sequence version number from FASTA.                                                                                 |
| `Length`                          | Amino acid length of the exact protein/isoform sequence                                                               |
| `Sequence`                        | Amino acid sequence for exact isoform.                                                                                 |
| `isoform_accession`               | Exact isoform-level UniProt ID. Same as `Entry`.                                                                            |
| `base_accession`                  | Canonical/base UniProt accession shared by all isoforms of the same protein/gene. Ex: `P04637-2 → P04637`.             |
| `is_canonical`                    | `True` if this row is the canonical/base protein sequence; `False` if alternative isoform.                                  |
| `base_prot_existence`             | Protein existence score inherited from the canonical/base protein. Useful since isoform-variants lack PE values in FASTA. |
| `prot_existence_filled`           | Protein existence score after filling missing isoform values using the base/canonical protein.                              |
| `Reviewed`                        | UniProt review status. Should all be `reviewed` for our Swiss-Prot dataset.                                                                   |
| `Entry Name`                      | UniProt human-readableentry name, e.g. `P53_HUMAN`.                                                         |
| `Protein names`                   | Full UniProt protein name.                                                                                                  |
| `Proteomes`                       | UniProt proteome annotation, usually showing human proteome/chromosome membership.                                          |
| `Protein existence`               | Text version of protein evidence level, e.g. “Evidence at protein level.”                                                   |
| `Protein families`                | UniProt protein family annotation.                                                                                          |
| `Alternative products (isoforms)` | UniProt annotation text about alternative products/isoforms for canonical proteins.                                    |
| `is_tf`                           | `True` if the base protein/gene is annotated as a transcription factor from the curated TF database.                        |
| `DBD`                             | DNA-binding domain family from the curated TF database. Non-TFs are labeled `Non-TF`.                                       |
| `tf_family`                       | General TF family label derived from `DBD`. Non-TFs are labeled `Non-TF`.                                                   |
| `tf_group`                        | Broad group: `TF` or `Non-TF`.                                                                                              |
| `idr_aaLen`                       | Total # of amino acids in predicted IDRs for this exact isoform. Sum across all IDR segments.                          |
| `n_idr_segments`                  | Number of predicted IDR segments in this isoform.                                                                           |
| `max_idr_len`                     | Length of the longest IDR segment in this isoform.                                                                          |
| `mean_idr_len`                    | Average length of IDR segments in this isoform.                                                                             |
| `mean_Rg_A`                       | Average radius of gyration across IDRs, in Ångstroms (length). Higher = more expanded IDR conformation; lower = more collapsed|
| `mean_Re_A`                       | Average distance between the first and last amino acid residues (the N & C terminal) of a polypeptide chain across IDRs, in Ångstroms|
| `mean_asphericity`                | Average IDR shape/asphericity. Higher values indicate more elongated/non-spherical ensembles.                               |
| `mean_scaling_e`                  | Average polymer scaling exponent across IDRs. Related to compactness/expansion of disordered chains.                        |
| `mean_prefactor`                  | Average polymer scaling prefactor from the IDR model. Technical polymer-fit parameter found in ML paper.                  |
| `mean_FCR`                        | Average fraction of charged residues in IDRs.                                                                               |
| `mean_NCPR`                       | Average net charge per residue in IDRs. Positive = more positively charged; negative = more negatively charged.     |
| `mean_kappa`                      | 0-1. Average charge patterning/segregation in IDRs. Lower (1) ~ + & - residues are well-mixed/evenly distributed; lower (0) ~ oppositely-charged residues are clustered (charges are segregated) |
| `mean_fract_neg`                  | Average fraction of - charged residues in IDRs. Usually D/E.                                                       |
| `mean_fract_pos`                  | Average fraction of + charged residues in IDRs. Usually K/R/H.                                                     |
| `mean_fract_aro`                  | Average fraction of aromatic residues in IDRs. Usually F/Y/W. Important for condensate-related interactions.**                |
| `mean_fract_pro`                  | Average fraction of proline residues in IDRs. Singificantly knownt to impact overall shape/physical state/compaction of disordered proteins.|
| `mean_fract_pol`                  | Average fraction of polar residues in IDRs.                                                                                 |
| `mean_fract_ali`                  | Average fraction of aliphatic residues in IDRs. These are typically hydrophobic amino acids: alanine, valine, isoleucine, and leucine. |
| `pct_idr`                         | Percent of this isoform predicted to be disordered: `idr_aaLen / Length * 100`. |

## Segment Redundant/Unwanted  Filters:
`is_tf`, `Protein existence`
## Add:
`pct_change_idr` : Percent IDR (`pct_idr`) change of all isoforms grouped to their respective gene (`base_accession`) to be disordered from max - min pct_idr across isoforms.

In [ ]:
# %% Clean/filter + add pct_change_idr
# Drop accidental index columns if present
all_isoforms_df = all_isoforms_df.drop(columns=["Unnamed: 0"], errors="ignore")
tf_isoforms_df = tf_isoforms_df.drop(columns=["Unnamed: 0"], errors="ignore")

# Drop redundant/unwanted columns
cols_to_drop = [
    "is_tf",
    "Protein existence",
]

all_isoforms_df = all_isoforms_df.drop(columns=cols_to_drop, errors="ignore")
tf_isoforms_df = tf_isoforms_df.drop(columns=cols_to_drop, errors="ignore")


# add pct_change_idr: max(pct_idr across isoforms of same base_accession) - min(pct_idr across isoforms of same base_accession)
def add_pct_change_idr(df):
    pct_change_df = (
        df
        .groupby("base_accession")["pct_idr"]
        .agg(
            min_pct_idr="min",
            max_pct_idr="max"
        )
        .reset_index()
    )
    pct_change_df["pct_change_idr"] = (
        pct_change_df["max_pct_idr"] - pct_change_df["min_pct_idr"]
    )
    df = df.merge(
        pct_change_df[["base_accession", "pct_change_idr"]],
        on="base_accession",
        how="left"
    )
    return df

all_isoforms_df = add_pct_change_idr(all_isoforms_df)
tf_isoforms_df = add_pct_change_idr(tf_isoforms_df)

# Checks
print("\nAll isoforms pct_change_idr summary:")
print(all_isoforms_df["pct_change_idr"].describe())
print("\nTF isoforms pct_change_idr summary:")
print(tf_isoforms_df["pct_change_idr"].describe())

# New w/ Susie Changes

In [ ]:
outdir = Path("../figures/proteome")
outdir.mkdir(parents=True, exist_ok=True)

def format_p(p): # function: format p-values
    if p < 1e-300:
        return "p < 1e-300"
    elif p < 0.001:
        return f"p = {p:.2e}"
    else:
        return f"p = {p:.3f}"


# Figure 1 gene-level violin: pct_change_idr is gene/base_accession-level
def violin_pct_change_idr(plot_df, title, filename=None, title_size=10, ylabel="Isoform-level disorder change (%IDR range)", figsize=(9, 5), save=True):
    """Violin plot: x-axis groups = Non-TF vs TF | one row = one gene/base protein"""
    # Split data
    non_tf_vals = plot_df.query("tf_group == 'Non-TF'")["pct_change_idr"].dropna()
    tf_vals = plot_df.query("tf_group == 'TF'")["pct_change_idr"].dropna()
    # P value stats
    _, pval = mannwhitneyu(tf_vals, non_tf_vals, alternative="two-sided")
    # Figure setup
    fig, ax = plt.subplots(figsize=figsize)
    positions = [1, 2]
    violin_width = 0.55
    max_half_width = violin_width / 2
    # Color style
    non_tf_fill = "#4C72B0"  # blue
    tf_fill = "#DD8452"      # orange
    non_tf_dark = "#2F4B7C"  # dark blue
    tf_dark = "#A95A2C"      # dark orange
    data = [non_tf_vals, tf_vals]
    fills = [non_tf_fill, tf_fill]
    darks = [non_tf_dark, tf_dark]
    # Helper: estimate violin width at y-value to get right median/quartile width sizes
    def kde_width_at_y(vals, y, max_half_width):
        vals = np.asarray(vals.dropna())
        if len(vals) < 2 or np.std(vals) == 0:
            return max_half_width * 0.20
        kde = gaussian_kde(vals)
        y_grid = np.linspace(vals.min(), vals.max(), 300)
        max_density = kde(y_grid).max()
        density_at_y = kde([y])[0]
        return max_half_width * (density_at_y / max_density)

    # Violin plot
    vp = ax.violinplot(data, positions=positions, widths=violin_width, showmeans=False, showmedians=False, showextrema=False)
    for body, fill in zip(vp["bodies"], fills):
        body.set_facecolor(fill)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(0.75)

    # Quartile + median lines matched to violin width
    for x, vals, dark in zip(positions, data, darks):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        for y in [q1, q3]:
            width_at_y = kde_width_at_y(vals, y, max_half_width)
            ax.hlines(y, x - width_at_y, x + width_at_y, linewidth=1.1, color=dark, linestyles="dashed")
        med_width = kde_width_at_y(vals, med, max_half_width)
        ax.hlines(med, x - med_width, x + med_width, linewidth=2.4, color=dark)

    # Labels / title / p-value
    ax.set_xticks(positions)
    ax.set_xticklabels([f"Non-TF\nn={len(non_tf_vals):,}", f"TF\nn={len(tf_vals):,}"])
    ax.set_xlim(0.45, 2.55)
    ax.set_ylim(-2, 102)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=title_size)

    legend_handles = [Patch(facecolor=non_tf_fill, edgecolor="black", alpha=0.75, label="Non-TF"),Patch(facecolor=tf_fill, edgecolor="black", alpha=0.75, label="TF"),]

    ax.legend(handles=legend_handles, frameon=True, edgecolor="black", facecolor="white", framealpha=1.0,loc="upper left", bbox_to_anchor=(1.03, 1.0), borderaxespad=0)
    ax.text(1.1, 0.85, format_p(pval), transform=ax.transAxes, ha="left", va="top", fontsize=9,bbox=dict(facecolor="white", edgecolor="black", alpha=1.0, boxstyle="round,pad=0.25"))
    plt.tight_layout(rect=[0, 0, 0.78, 1])

    # Save file
    if save and filename is not None:
        plt.savefig(outdir / f"{filename}.pdf", bbox_inches="tight")
    plt.show()

    # Print summary
    print(title)
    print(f"Non-TF: n={len(non_tf_vals):,}, median={non_tf_vals.median():.3f}, mean={non_tf_vals.mean():.3f}")
    print(f"TF:     n={len(tf_vals):,}, median={tf_vals.median():.3f}, mean={tf_vals.mean():.3f}")
    print(format_p(pval))

In [ ]:
# Gene-level dataframe for Fig. 1:
canonical_gene_df = (all_isoforms_df.query("is_canonical == True").drop_duplicates("base_accession").copy())
print("Canonical/base protein rows:", canonical_gene_df.shape[0])
print("Unique base proteins:", canonical_gene_df["base_accession"].nunique())
print("TF group counts:")
print(canonical_gene_df["tf_group"].value_counts())

# ------------------------------------------------------------
# Figure 1A: gene-level pct_change_idr, all genes
# ------------------------------------------------------------

fig1a_df = canonical_gene_df.copy()

print("\nFigure 1A genes/base proteins:", fig1a_df.shape[0])
print("Unique base proteins:", fig1a_df["base_accession"].nunique())
print("TF group counts:")
print(fig1a_df["tf_group"].value_counts())

violin_pct_change_idr(
    plot_df=fig1a_df,
    title="TFs vs. Non-TFs Isoform-Level Disorder Variation",
    title_size=9,
    filename="fig1A_pct_change_idr",
    ylabel="Isoform-level disorder change (Δ% IDR variation)")

In [ ]:
# Make a copy so the original fig1a_df stays unchanged
fig1a_quartile_df = fig1a_df.dropna(subset=["pct_change_idr", "tf_group"]).copy()

# Assign quartiles separately within TFs and Non-TFs
fig1a_quartile_df["pct_change_idr_quartile"] = (
    fig1a_quartile_df.groupby("tf_group")["pct_change_idr"].transform(
        lambda x: pd.qcut(
            x.rank(method="first"),q=4,labels=["Q1", "Q2", "Q3", "Q4"])))

fig1a_quartile_df[["base_accession","gene_name","tf_group","pct_change_idr","pct_change_idr_quartile"]].sort_values(["tf_group", "pct_change_idr"],ascending=[True, False]).head(10)

fig1a_quartile_df.to_csv("../data/annotated/fig1a_quartiles.csv",index=False)

fig1a_top_quartile_df = fig1a_quartile_df.query(
    "pct_change_idr_quartile == 'Q4'"
).copy()

print(fig1a_top_quartile_df["tf_group"].value_counts())

In [ ]:
# ------------------------------------------------------------
# Figure 1B: gene-level pct_change_idr, genes where any isoform has >=1 IDR
# ------------------------------------------------------------

base_accessions_with_any_idr = (all_isoforms_df.groupby("base_accession")["n_idr_segments"].max().loc[lambda x: x > 0].index)

fig1b_df = (canonical_gene_df.query("base_accession in @base_accessions_with_any_idr").copy())

print("\nFigure 1B genes/base proteins:", fig1b_df.shape[0])
print("Unique base proteins:", fig1b_df["base_accession"].nunique())
print("TF group counts:")
print(fig1b_df["tf_group"].value_counts())

violin_pct_change_idr(
    plot_df=fig1b_df,
    title="TFs vs. Non-TFs Isoform-Level Disorder Variation Among IDR-Containing Genes",
    title_size=9,
    filename="fig1B_pct_change_idr_WITHisoforms",
    ylabel="Isoform-level disorder change (%IDR range)")

In [ ]:
# Figure 2 split-half violin: pct_idr is isoform-level, so we need violin splits
def split_violin_tf_vs_nontf(plot_df, metric, title,filename=None,title_size=10,ylabel=None,figsize=(8, 5),save=True,):
    """Split-half violin plot: x-axis groups = Non-TF vs TF | left half = canonical rows only | right half = all isoform rows"""
    # Split data
    non_tf_canon = (plot_df.query("tf_group == 'Non-TF' and is_canonical == True")[metric].dropna())
    tf_canon = (plot_df.query("tf_group == 'TF' and is_canonical == True")[metric].dropna())
    non_tf_all = (plot_df.query("tf_group == 'Non-TF'")[metric].dropna())
    tf_all = (plot_df.query("tf_group == 'TF'")[metric].dropna())
    # P value Stats
    # Across groups: Non-TF vs TF
    _, p_canon_non_tf_vs_tf = mannwhitneyu(non_tf_canon,tf_canon,alternative="two-sided")
    _, p_all_non_tf_vs_tf = mannwhitneyu(non_tf_all,tf_all,alternative="two-sided")
    # Within groups: canonical/blue vs all-isoform/orange
    _, p_non_tf_blue_vs_orange = mannwhitneyu(non_tf_canon,non_tf_all,alternative="two-sided")
    _, p_tf_blue_vs_orange = mannwhitneyu(tf_canon,tf_all,alternative="two-sided")
    # Figure setup
    fig, ax = plt.subplots(figsize=figsize)
    centers = np.array([1, 2])
    # Space between canonical and all-isoform halves
    xgap = 0.05
    canon_positions = centers - xgap / 2
    all_positions = centers + xgap / 2
    violin_width = 0.55
    max_half_width = violin_width / 2
    # Color Style
    canon_fill = "#4C72B0"   # blue
    all_fill = "#DD8452"     # orange
    canon_dark = "#2F4B7C"   # dark blue
    all_dark = "#A95A2C"     # dark orange
    # Helper: estimate violin width at y-value to get right median/quartile width sizes 
    def kde_width_at_y(vals, y, max_half_width):
        vals = np.asarray(vals.dropna())
        if len(vals) < 2 or np.std(vals) == 0:
            return max_half_width * 0.20
        kde = gaussian_kde(vals)
        y_grid = np.linspace(vals.min(), vals.max(), 300)
        max_density = kde(y_grid).max()
        density_at_y = kde([y])[0]
        return max_half_width * (density_at_y / max_density)
    # Left half = canonical only
    vp_left = ax.violinplot([non_tf_canon, tf_canon],positions=canon_positions,widths=violin_width,showmeans=False,showmedians=False,showextrema=False,side="low")
    for body in vp_left["bodies"]:
        body.set_facecolor(canon_fill)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(0.75)
    # Right half = all isoforms
    vp_right = ax.violinplot([non_tf_all, tf_all],positions=all_positions,widths=violin_width,showmeans=False,showmedians=False,showextrema=False,side="high")
    for body in vp_right["bodies"]:
        body.set_facecolor(all_fill)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(0.75)
    # Quartile + median lines matched to violin width
    # Canonical half, left side
    for x, vals in zip(canon_positions, [non_tf_canon, tf_canon]):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        for y in [q1, q3]:
            width_at_y = kde_width_at_y(vals, y, max_half_width)
            ax.hlines(y,x - width_at_y,x,linewidth=1.1,color=canon_dark,linestyles="dashed")
        med_width = kde_width_at_y(vals, med, max_half_width)
        ax.hlines(med,x - med_width,x,linewidth=2.4,color=canon_dark)
    # All isoforms half, right side
    for x, vals in zip(all_positions, [non_tf_all, tf_all]):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        for y in [q1, q3]:
            width_at_y = kde_width_at_y(vals, y, max_half_width)
            ax.hlines(y,x,x + width_at_y,linewidth=1.1,color=all_dark,linestyles="dashed")
        med_width = kde_width_at_y(vals, med, max_half_width)
        ax.hlines(med,x,x + med_width,linewidth=2.4,color=all_dark)
    # Labels / title / p-values
    ax.set_xticks(centers)
    ax.set_xticklabels([f"Non-TF\ncanon={len(non_tf_canon):,}\nall={len(non_tf_all):,}",f"TF\ncanon={len(tf_canon):,}\nall={len(tf_all):,}",])
    ax.set_xlim(0.45, 2.55)
    ax.set_ylim(-2, 102)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=title_size)
    legend_handles = [Patch(facecolor=canon_fill, edgecolor="black", alpha=0.75, label="Canonical only"),Patch(facecolor=all_fill, edgecolor="black", alpha=0.75, label="All isoforms"),]
    ax.legend(handles=legend_handles,frameon=True,edgecolor="black",facecolor="white",framealpha=1.0,loc="upper left",bbox_to_anchor=(1.03, 1.0),borderaxespad=0)
    ax.text(1.03, 0.78,f"Across groups:\n"
        f"canonical Non-TF vs TF: {format_p(p_canon_non_tf_vs_tf)}\n"
        f"all isoforms Non-TF vs TF: {format_p(p_all_non_tf_vs_tf)}\n\n"
        f"Within groups:\n"
        f"Non-TF blue vs orange: {format_p(p_non_tf_blue_vs_orange)}\n"
        f"TF blue vs orange: {format_p(p_tf_blue_vs_orange)}",
        transform=ax.transAxes,ha="left",va="top",fontsize=8.5,
        bbox=dict(facecolor="white",edgecolor="black",alpha=1.0,boxstyle="round,pad=0.25"))
    fig.subplots_adjust(left=0.12, right=0.76, top=0.88, bottom=0.18)
    # Save file
    if save and filename is not None:
        plt.savefig(outdir / f"{filename}.pdf", bbox_inches="tight")
    plt.show()
    # Print summary
    print(title)
    
    print("Across-group comparisons:")
    print("Canonical Non-TF vs canonical TF:")
    print(f"  Non-TF canonical: n={len(non_tf_canon):,}, median={non_tf_canon.median():.3f}, mean={non_tf_canon.mean():.3f}")
    print(f"  TF canonical:     n={len(tf_canon):,}, median={tf_canon.median():.3f}, mean={tf_canon.mean():.3f}")
    print(f"  {format_p(p_canon_non_tf_vs_tf)}")
    
    print("All-isoform Non-TF vs all-isoform TF:")
    print(f"  Non-TF all: n={len(non_tf_all):,}, median={non_tf_all.median():.3f}, mean={non_tf_all.mean():.3f}")
    print(f"  TF all:     n={len(tf_all):,}, median={tf_all.median():.3f}, mean={tf_all.mean():.3f}")
    print(f"  {format_p(p_all_non_tf_vs_tf)}")
    
    print("\nWithin-group blue vs orange comparisons:")
    print("Non-TF canonical vs all isoforms:")
    print(f"  Canonical only: n={len(non_tf_canon):,}, median={non_tf_canon.median():.3f}, mean={non_tf_canon.mean():.3f}")
    print(f"  All isoforms:   n={len(non_tf_all):,}, median={non_tf_all.median():.3f}, mean={non_tf_all.mean():.3f}")
    print(f"  {format_p(p_non_tf_blue_vs_orange)}")
    
    print("TF canonical vs all isoforms:")
    print(f"  Canonical only: n={len(tf_canon):,}, median={tf_canon.median():.3f}, mean={tf_canon.mean():.3f}")
    print(f"  All isoforms:   n={len(tf_all):,}, median={tf_all.median():.3f}, mean={tf_all.mean():.3f}")
    print(f"  {format_p(p_tf_blue_vs_orange)}")

In [ ]:
# ------------------------------------------------------------
# Figure 2A: isoform-level pct_idr, all isoforms
# ------------------------------------------------------------

split_violin_tf_vs_nontf(
    plot_df=all_isoforms_df,
    metric="pct_idr",
    title="TFs vs. Non-TFs Isoform Disorder Content",
    title_size=9,
    filename="fig2A_split_pct_idr_all_isoforms",
    ylabel="%IDR")

# ------------------------------------------------------------
# Figure 2B: isoform-level pct_idr, IDR-containing isoforms only
# ------------------------------------------------------------

fig2b_df = all_isoforms_df.query("n_idr_segments > 0").copy()

print("\nFigure 2B isoform rows:", fig2b_df.shape[0])
print("TF group counts:")
print(fig2b_df["tf_group"].value_counts())

split_violin_tf_vs_nontf(
    plot_df=fig2b_df,
    metric="pct_idr",
    title="TFs vs. Non-TFs Isoform Disorder Content Among IDR-Containing Isoforms",
    title_size=9,
    filename="fig2B_split_pct_idr_WITHisoforms",
    ylabel="%IDR")

## Add Evolutionary Annotations

In [ ]:
# remove genes < 2 proteins made (has to be caonical + 2 isoforms atleast)
# filter for proteins with isoforms > 5 -> have been more alternatively spliced
# isoform-level gene age differences 

# look into database, has isoform-level annotations -> https://tfisodb.org/about.html
# check PPI interactions and see whether # of interactinos with differne transcription proteins increases with disorder
# also check with their predicted structures to see whether metapredict and their 'alphafold' structures (known to be less accurate) how similar their two sequences are.

In [ ]:
# Load GenOrigin gene age annotations
# GenOrigin file: evolutionary age information for human genes
# downloaded from http://chenzxlab.hzau.edu.cn/GenOrigin/#!/download
genorigin_path = ("../data/raw/genorigin/Homo_sapiens.csv")
genorigin_df = pd.read_csv(genorigin_path).sort_values("gene_age")

# Load ID mapping file
# This file maps Ensembl gene IDs -> UniProt accessions.
# GenOrigin uses Ensembl gene IDs, isoform files use UniProt accessions.
HPC_ENSG_TO_UPKB = "/scratch/gpfs/AKEY/ssong/tf_idr_paper/data/idmapping/ensg_to_upkb.parquet"
bins = [0, 100, 500, 1000, np.inf]
labels = ["<100", "100-500", "500-1000", ">1000"]
AGE_SNAPSHOT = EXTERNAL / "hpc_snapshot" / "gene_age_from_hpc_mapping.csv"

if Path(HPC_ENSG_TO_UPKB).exists():
    # Original derivation (AKEY cluster)
    idmapping_path = HPC_ENSG_TO_UPKB
    idmapping = pd.read_parquet(idmapping_path,engine="fastparquet")[["ENSG", "upkb_accession"]]

    # Merge GenOrigin gene age data with UniProt IDs accessions
    age_map = (idmapping.merge(genorigin_df,left_on="ENSG",right_on="ensembl_gene_id",how="inner").drop(["ENSG", "ensembl_gene_id"], axis=1))

    # Clean gene_age column | group genes labeled as ">4290". -> 4300
    age_map.loc[age_map["gene_age"] == ">4290", "gene_age"] = "4300"  # str: column is string dtype
    age_map["gene_age"] = age_map["gene_age"].astype(int) # Convert gene_age to integer.

    # Create broader age bins to group genes into evolutionary age categories.
    age_map["age_bin"] = pd.cut(age_map["gene_age"],bins=bins, labels=labels, right=False)
    # Keep only columns needed for merging into your master files
    age_map_clean = (
        age_map[["upkb_accession", "gene_age", "age_bin"]]
        .drop_duplicates())

    # Ensure one gene-age annotation per UniProt accession
    age_map_unique = age_map_clean.drop_duplicates("upkb_accession")
else:
    # Off-cluster: per-gene result of exactly the derivation above, frozen from the cluster run.
    age_map_unique = pd.read_csv(AGE_SNAPSHOT)
    age_map_unique["age_bin"] = pd.Categorical(age_map_unique["age_bin"], categories=labels, ordered=True)


# Merge gene age annotations onto all isoforms
# Gene age is gene-level, so every isoform from the same base_accession gets the same gene_age.
all_isoforms_df_gene_age = all_isoforms_df.merge(age_map_unique,left_on="base_accession",right_on="upkb_accession",how="left",validate="many_to_one")

# Merge gene age annotations onto TF isoforms only
tf_isoforms_df_gene_age = tf_isoforms_df.merge(age_map_unique,left_on="base_accession",right_on="upkb_accession",how="left",validate="many_to_one")

## Add # of Isoforms per Gene

In [ ]:
# Remove old n_isoforms columns if they already exist
all_isoforms_df_gene_age = all_isoforms_df_gene_age.drop(
    columns=[col for col in all_isoforms_df_gene_age.columns if col.startswith("n_isoforms")],
    errors="ignore")
tf_isoforms_df_gene_age = tf_isoforms_df_gene_age.drop(
    columns=[col for col in tf_isoforms_df_gene_age.columns if col.startswith("n_isoforms")],
    errors="ignore")

# Count number of isoforms per gene for ALL proteins
all_isoform_counts = (all_isoforms_df_gene_age.groupby("base_accession")["isoform_accession"].nunique().reset_index(name="n_isoforms"))
all_isoforms_df_gene_age = all_isoforms_df_gene_age.merge(all_isoform_counts,on="base_accession",how="left",validate="many_to_one")
# Count number of isoforms per gene for TF proteins only
tf_isoform_counts = (tf_isoforms_df_gene_age.groupby("base_accession")["isoform_accession"].nunique().reset_index(name="n_isoforms"))
tf_isoforms_df_gene_age = tf_isoforms_df_gene_age.merge(tf_isoform_counts,on="base_accession",how="left",validate="many_to_one")

## Add Mean IDR % grouped by Gene

In [ ]:
# Remove old mean_pct_idr columns if they already exist (re-running must not create _x/_y duplicates)
all_isoforms_df_gene_age = all_isoforms_df_gene_age.drop(
    columns=[c for c in all_isoforms_df_gene_age.columns if c.startswith("mean_pct_idr_all_isoforms")])
tf_isoforms_df_gene_age = tf_isoforms_df_gene_age.drop(
    columns=[c for c in tf_isoforms_df_gene_age.columns if c.startswith("mean_pct_idr_all_isoforms")])

# Calculate mean % IDR across all isoforms per gene for ALL proteins
all_mean_pct_idr = (all_isoforms_df_gene_age.groupby("base_accession")["pct_idr"].mean().reset_index(name="mean_pct_idr_all_isoforms"))
all_isoforms_df_gene_age = all_isoforms_df_gene_age.merge(all_mean_pct_idr,on="base_accession",how="left",validate="many_to_one")
# Calculate mean % IDR across all isoforms per gene for TF proteins only
tf_mean_pct_idr = (tf_isoforms_df_gene_age.groupby("base_accession")["pct_idr"].mean().reset_index(name="mean_pct_idr_all_isoforms"))
tf_isoforms_df_gene_age = tf_isoforms_df_gene_age.merge(tf_mean_pct_idr,on="base_accession",how="left",validate="many_to_one")

# Save new master files
all_isoforms_df_gene_age.to_csv("../data/annotated/IDRisoforms_df_geneage.csv",index=False)
tf_isoforms_df_gene_age.to_csv("../data/annotated/TranscriptionIsoforms_df_geneage.csv",index=False)

### NEW:

In [ ]:
# ------------------------------------------------------------
# Make gene-level dataframes
# ------------------------------------------------------------

all_gene_level_df = (all_isoforms_df_gene_age.drop_duplicates("base_accession")
    .dropna(subset=["n_isoforms", "mean_pct_idr_all_isoforms", "pct_change_idr", "tf_group"]).copy())
tf_gene_level_df = (tf_isoforms_df_gene_age.drop_duplicates("base_accession")
    .dropna(subset=["n_isoforms", "mean_pct_idr_all_isoforms", "pct_change_idr"]).copy())

tf_gene_level_df["tf_group"] = "TF"
non_tf_gene_level_df = (all_gene_level_df.query("tf_group == 'Non-TF'").copy())

In [ ]:
# sns.stripplot(jitter=True) draws jitter from NumPy's global RNG; seed it so point placement
# is identical on every run (display only; no statistic depends on it).
np.random.seed(0)

# ------------------------------------------------------------
# One combined figure:
# rows = TF / All / Non-TF
# col 1 = mean_pct_idr_all_isoforms
# col 2 = pct_change_idr
# ------------------------------------------------------------

isoform_bins = [2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, float("inf")]
isoform_bin_labels = ["3", "4", "5", "6", "7", "8", "9", "10+"]

# Clean copies
tf_plot_df = tf_gene_level_df.copy()
all_plot_df = all_gene_level_df.copy()
non_tf_plot_df = all_gene_level_df.query("tf_group == 'Non-TF'").copy()

# Correct binning:
# 3 = exactly 3 isoforms, 4 = exactly 4 isoforms, ..., 10+ = 10 or more
for df in [tf_plot_df, all_plot_df, non_tf_plot_df]:
    df["isoform_count_bin"] = pd.cut(
        df["n_isoforms"],
        bins=isoform_bins,
        labels=isoform_bin_labels
    )

# Set up figure: 3 rows x 2 columns
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True)

row_specs = [
    (tf_plot_df, "palegreen", "TF genes"),
    (all_plot_df, "lightgray", "All genes"),
    (non_tf_plot_df, "mistyrose", "Non-TF genes")
]

col_specs = [
    ("mean_pct_idr_all_isoforms", "Mean % IDR across isoforms"),
    ("pct_change_idr", "Δ% IDR across isoforms")]

for i, (df, color, row_title) in enumerate(row_specs):
    for j, (metric, ylab) in enumerate(col_specs):
        ax = axes[j, i]
        sub_df = df.dropna(subset=["isoform_count_bin", metric]).copy()
        sns.boxplot(data=sub_df,x="isoform_count_bin",y=metric,
            order=isoform_bin_labels,ax=ax,color=color,showfliers=False)
        sns.stripplot(data=sub_df,x="isoform_count_bin",y=metric,
            order=isoform_bin_labels,ax=ax,color="black",alpha=0.35,size=2,jitter=True)
        # Titles only on top row
        if i == 1: ax.set_title(ylab, fontsize=12)
        # Y labels only on left edge of each subplot
        ax.set_ylabel(f"{row_title}\n{ylab}", fontsize=10)
        # X label only on bottom row
        if i == 1: ax.set_xlabel("Number of isoforms per gene")
        else: ax.set_xlabel("")

fig.suptitle("Isoforms vs mean % IDR and Δ% IDR variation",fontsize=15,y=1.02)
plt.tight_layout()
plt.savefig(outdir / "fig3A_combined_mean_and_range_pct_idr.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Build clean plotting dataframe
# ------------------------------------------------------------
plot_df = all_gene_level_df.copy()

# Keep only genes with enough isoforms to discuss isoform-level variation
plot_df = plot_df.dropna(subset=["base_accession","tf_group","n_isoforms","mean_pct_idr_all_isoforms","pct_change_idr"])
plot_df = plot_df[plot_df["n_isoforms"] >= 3].copy()

# Make TF status numeric for regression
plot_df["is_tf"] = plot_df["tf_group"].eq("TF").astype(int)

# Optional: cap x-axis for visualization only
# Keeps genes with 10+ isoforms visible but prevents extreme x-axis stretching
plot_df["n_isoforms_plot"] = plot_df["n_isoforms"].clip(upper=10)

# Add jitter so points at 3,4,5,... don't sit exactly on top of each other
rng = np.random.default_rng(42)
plot_df["n_isoforms_jitter"] = (
    plot_df["n_isoforms_plot"] +
    rng.normal(0, 0.08, size=len(plot_df))
)

# Helper function for regression line
# ------------------------------------------------------------
def add_regression_line(ax, data, x_col, y_col, label):
    x = data[x_col].values
    y = data[y_col].values
    if len(data) < 3:
        return
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, linewidth=2, label=label)

# Plot A and Plot B side by side
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

groups = ["Non-TF", "TF"]

# Plot A: n_isoforms vs mean %IDR
for group in groups:
    subset = plot_df[plot_df["tf_group"] == group]
    axes[0].scatter(
        subset["n_isoforms_jitter"],subset["mean_pct_idr_all_isoforms"],
        alpha=0.25,s=15,label=group)
    add_regression_line(
        axes[0],subset,"n_isoforms_plot",
        "mean_pct_idr_all_isoforms",f"{group} regression")

axes[0].set_title("Number of isoforms vs mean %IDR")
axes[0].set_xlabel("Number of isoforms per gene")
axes[0].set_ylabel("Mean %IDR across isoforms")
axes[0].set_xticks(range(3, 11))
axes[0].set_xticklabels(["3", "4", "5", "6", "7", "8", "9", "10+"])
axes[0].legend(frameon=False)

# Plot B: n_isoforms vs ΔIDR
for group in groups:
    subset = plot_df[plot_df["tf_group"] == group]

    axes[1].scatter(subset["n_isoforms_jitter"],subset["pct_change_idr"],
        alpha=0.25,s=15,label=group)

    add_regression_line(axes[1],subset,
        "n_isoforms_plot","pct_change_idr",f"{group} regression")

axes[1].set_title("Number of isoforms vs isoform-level Δ %IDR")
axes[1].set_xlabel("Number of isoforms per gene")
axes[1].set_ylabel("ΔIDR: %IDR range across isoforms")
axes[1].set_xticks(range(3, 11))
axes[1].set_xticklabels(["3", "4", "5", "6", "7", "8", "9", "10+"])
axes[1].legend(frameon=False)

plt.tight_layout()

fig.savefig(outdir / "fig3B_meanidr_rangeidr_linearregress.pdf",bbox_inches="tight")

plt.show()

In [ ]:
age_order = ["<100", "100-500", "500-1000", ">1000"]

def split_violin_by_age_tf_vs_nontf(plot_df,metric,title,ylabel,filename=None,figsize=(9, 5),save=True):
    """
    Split violin plot by age_bin:
    x-axis = gene age bin
    left half = Non-TF
    right half = TF
    y-axis = metric
    """

    plot_df = plot_df.dropna(subset=["age_bin", "tf_group", metric]).copy()
    plot_df["age_bin"] = pd.Categorical(plot_df["age_bin"], categories=age_order, ordered=True)

    fig, ax = plt.subplots(figsize=figsize)
    centers = np.arange(1, len(age_order) + 1)

    xgap = 0.05
    non_tf_positions = centers - xgap / 2
    tf_positions = centers + xgap / 2

    violin_width = 0.75
    max_half_width = violin_width / 2

    non_tf_fill = "#4C72B0"
    tf_fill = "#DD8452"
    non_tf_dark = "#2F4B7C"
    tf_dark = "#A95A2C"

    pvals = {}

    def kde_width_at_y(vals, y, max_half_width):
        vals = np.asarray(pd.Series(vals).dropna())
        if len(vals) < 2 or np.std(vals) == 0:
            return max_half_width * 0.20
        kde = gaussian_kde(vals)
        y_grid = np.linspace(vals.min(), vals.max(), 300)
        max_density = kde(y_grid).max()
        density_at_y = kde([y])[0]
        return max_half_width * (density_at_y / max_density)

    def draw_quartile_median_lines(x, vals, side, dark):
        vals = pd.Series(vals).dropna()
        if len(vals) == 0:
            return

        q1, med, q3 = np.percentile(vals, [25, 50, 75])

        for y in [q1, q3]:
            width_at_y = kde_width_at_y(vals, y, max_half_width)
            if side == "left":
                ax.hlines(y, x - width_at_y, x, linewidth=1.1, color=dark, linestyles="dashed")
            else:
                ax.hlines(y, x, x + width_at_y, linewidth=1.1, color=dark, linestyles="dashed")

        med_width = kde_width_at_y(vals, med, max_half_width)
        if side == "left":
            ax.hlines(med, x - med_width, x, linewidth=2.4, color=dark)
        else:
            ax.hlines(med, x, x + med_width, linewidth=2.4, color=dark)

    for i, age in enumerate(age_order):
        non_tf_x = non_tf_positions[i]
        tf_x = tf_positions[i]

        non_tf_vals = plot_df.query("age_bin == @age and tf_group == 'Non-TF'")[metric].dropna()
        tf_vals = plot_df.query("age_bin == @age and tf_group == 'TF'")[metric].dropna()

        if len(non_tf_vals) >= 2:
            vp_left = ax.violinplot([non_tf_vals],positions=[non_tf_x],widths=violin_width,showmeans=False,showmedians=False,showextrema=False,side="low")
            for body in vp_left["bodies"]:
                body.set_facecolor(non_tf_fill)
                body.set_edgecolor("black")
                body.set_linewidth(1.0)
                body.set_alpha(0.75)
            draw_quartile_median_lines(non_tf_x, non_tf_vals, "left", non_tf_dark)

        if len(tf_vals) >= 2:
            vp_right = ax.violinplot([tf_vals],positions=[tf_x],widths=violin_width,showmeans=False,showmedians=False,showextrema=False,side="high")
            for body in vp_right["bodies"]:
                body.set_facecolor(tf_fill)
                body.set_edgecolor("black")
                body.set_linewidth(1.0)
                body.set_alpha(0.75)
            draw_quartile_median_lines(tf_x, tf_vals, "right", tf_dark)

        if len(non_tf_vals) > 0 and len(tf_vals) > 0:
            _, pval = mannwhitneyu(non_tf_vals, tf_vals, alternative="two-sided")
            pvals[age] = pval
        else:
            pvals[age] = np.nan

    ax.set_xticks(centers)
    ax.set_xticklabels(age_order)
    ax.set_xlabel("Gene age bin")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    legend_handles = [Patch(facecolor=non_tf_fill, edgecolor="black", alpha=0.75, label="Non-TF"),Patch(facecolor=tf_fill, edgecolor="black", alpha=0.75, label="TF")]
    ax.legend(handles=legend_handles,frameon=True,edgecolor="black",facecolor="white",framealpha=1.0,loc="upper left",bbox_to_anchor=(1.03, 1.0),borderaxespad=0)

    p_text = "\n".join([f"{age}: {format_p(p)}" if not pd.isna(p) else f"{age}: NA" for age, p in pvals.items()])
    ax.text(1.03,0.7,p_text,transform=ax.transAxes,ha="left",va="top",fontsize=8.5,bbox=dict(facecolor="white",edgecolor="black",boxstyle="round,pad=0.25"))

    plt.tight_layout(rect=[0, 0, 0.82, 1])

    if save and filename is not None:
        plt.savefig(outdir / f"{filename}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
isoform_level_age_df = (all_isoforms_df_gene_age.dropna(subset=["age_bin", "tf_group", "pct_idr"]).copy())
split_violin_by_age_tf_vs_nontf(plot_df=isoform_level_age_df,metric="pct_idr",title="% IDR by Gene Age: TFs vs Non-TFs",ylabel="%IDR",filename="fig4A_geneage_pct_idr")

In [ ]:
def full_violin_by_age_tf_vs_nontf(plot_df,metric,title,ylabel,filename=None,figsize=(9, 5),save=True):
    age_order = ["<100", "100-500", "500-1000", ">1000"]
    plot_df = plot_df.dropna(subset=["age_bin", "tf_group", metric]).copy()
    plot_df["age_bin"] = pd.Categorical(plot_df["age_bin"],categories=age_order,ordered=True)

    fig, ax = plt.subplots(figsize=figsize)

    sns.violinplot(data=plot_df,x="age_bin",y=metric,hue="tf_group",order=age_order,hue_order=["Non-TF", "TF"],cut=0,inner="quartile",density_norm="width",ax=ax)
    ax.set_xlabel("Gene age bin")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.legend(title="Gene group",frameon=True,edgecolor="black",facecolor="white",framealpha=1.0,loc="upper left",bbox_to_anchor=(1.03, 1.0))
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    if save and filename is not None:
        plt.savefig(outdir / f"{filename}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
gene_level_age_df = (all_isoforms_df_gene_age.drop_duplicates("base_accession").dropna(subset=["age_bin", "tf_group", "n_isoforms"]).copy())
full_violin_by_age_tf_vs_nontf(plot_df=gene_level_age_df,metric="n_isoforms",title="Number of Isoforms by Gene Age: TFs vs Non-TFs",ylabel="Number of isoforms per gene",filename="fig4B_geneage_nisoforms")

In [ ]:
# ------------------------------------------------------------
# Build clean TF-family grouping
# ------------------------------------------------------------

family_df = all_isoforms_df.copy()

# Create is_tf if it does not exist
if "is_tf" not in family_df.columns:
    if "tf_group" in family_df.columns:
        family_df["is_tf"] = family_df["tf_group"].eq("TF")
    else:
        tf_base_accessions = set(tf_isoforms_df["base_accession"].dropna())
        family_df["is_tf"] = family_df["base_accession"].isin(tf_base_accessions)

# Make sure is_tf is boolean
family_df["is_tf"] = family_df["is_tf"].astype(bool)

# Fill missing TF family labels
family_df["tf_family"] = family_df["tf_family"].fillna("Unknown")

# Count unique isoforms per gene
n_isoforms_per_gene = (family_df.groupby("base_accession")["isoform_accession"].nunique().reset_index(name="n_isoforms"))

# One row per gene with TF status and TF family
gene_family_info = (family_df.drop_duplicates("base_accession")[["base_accession", "gene_name", "is_tf", "tf_family"]].copy())

gene_level_df = gene_family_info.merge(n_isoforms_per_gene,on="base_accession",how="left",validate="one_to_one")

# Top 7 TF families, excluding Unknown
# Rank by number of genes; break ties by number of isoforms, then name (deterministic).
_fam = (gene_level_df
    .query("is_tf == True and tf_family != 'Unknown'")
    .groupby("tf_family").agg(n_genes=("base_accession", "nunique"), n_isoforms=("n_isoforms", "sum"))
    .reset_index()
    .sort_values(["n_genes", "n_isoforms", "tf_family"], ascending=[False, False, True]))
top_7_tf_families = _fam["tf_family"].head(7).tolist()

# Create plotting labels: top 7 TF families + Unknown TFs + Other TF families + Non-TFs
gene_level_df["family_plot"] = np.where(gene_level_df["is_tf"] == False,"Non-TF",
    np.where(gene_level_df["tf_family"].isin(top_7_tf_families),
        gene_level_df["tf_family"], np.where(gene_level_df["tf_family"] == "Unknown", "Unknown", "Other")))

family_order = top_7_tf_families + ["Unknown", "Other", "Non-TF"]

print("Family order:")
print(family_order)

print("\nGene counts per group:")
print(gene_level_df["family_plot"].value_counts().reindex(family_order))

# ------------------------------------------------------------
# Add gene-level IDR summary metrics
# ------------------------------------------------------------

gene_idr_metrics = (
    family_df
    .dropna(subset=["base_accession", "pct_idr"])
    .groupby("base_accession")
    .agg(
        min_pct_idr=("pct_idr", "min"),
        max_pct_idr=("pct_idr", "max"),
        mean_pct_idr_all_isoforms=("pct_idr", "mean"),
        pct_change_idr=("pct_idr", lambda x: x.max() - x.min())
    )
    .reset_index()
)

gene_level_df = gene_level_df.merge(
    gene_idr_metrics,
    on="base_accession",
    how="left",
    validate="one_to_one"
)

print(gene_level_df.columns)
print(gene_level_df[["base_accession", "n_isoforms", "pct_change_idr"]].head())
# ------------------------------------------------------------
# Build isoform-level dataframe for IDR metrics
# ------------------------------------------------------------

isoform_family_df = family_df.copy()

isoform_family_df["family_plot"] = np.where(
    isoform_family_df["is_tf"] == False,
    "Non-TF",
    np.where(isoform_family_df["tf_family"].isin(top_7_tf_families),
        isoform_family_df["tf_family"],np.where(isoform_family_df["tf_family"] == "Unknown", "Unknown", "Other")))

isoform_family_df = isoform_family_df.query("family_plot in @family_order").copy()

print("\nIsoform counts per group:")
print(isoform_family_df["family_plot"].value_counts().reindex(family_order))

In [ ]:
def plot_family_metric_histograms(
    plot_df, family_order, metric, xlabel, title, filename=None, bins=None,
    color_tf="#4C72B0", color_nontf="darkred",
    obs_label="genes", mean_suffix="", save=True, show_legend=False
):
    fig, axes = plt.subplots(2, 5, figsize=(15, 5), sharex=True, sharey=False)
    axes = axes.flatten()

    plot_df = plot_df.dropna(subset=["family_plot", "is_tf", metric]).copy()

    if bins is None:
        vals = plot_df[metric].dropna()
        if vals.nunique() <= 30 and np.all(vals == vals.astype(int)):
            bins = np.arange(vals.min() - 0.5, vals.max() + 1.5, 1)
        else:
            bins = 20

    # Reference distributions
    non_tf_ref_vals = plot_df.query("family_plot == 'Non-TF'")[metric].dropna()
    all_tf_ref_vals = plot_df.query("is_tf == True")[metric].dropna()

    non_tf_ref_mean = non_tf_ref_vals.mean()
    all_tf_ref_mean = all_tf_ref_vals.mean()

    for ax, family in zip(axes, family_order):
        family_vals = plot_df.query("family_plot == @family")[metric].dropna()

        n_obs = len(family_vals)
        mean_val = family_vals.mean()

        # P-value logic + reference line logic
        if family == "Non-TF":
            comparison_vals = all_tf_ref_vals
            comparison_label = "vs all TFs"
            ref_mean = all_tf_ref_mean
        else:
            comparison_vals = non_tf_ref_vals
            comparison_label = "vs Non-TF"
            ref_mean = non_tf_ref_mean

        if len(family_vals) > 0 and len(comparison_vals) > 0:
            _, pval = mannwhitneyu(
                family_vals,
                comparison_vals,
                alternative="two-sided"
            )
            p_label = f"{comparison_label}\n{format_p(pval)}"
        else:
            p_label = "P = NA"

        color = color_nontf if family == "Non-TF" else color_tf

        ax.hist(
            family_vals,
            bins=bins,
            color=color,
            edgecolor="white",
            alpha=0.8
        )

        # Black dashed line = current panel mean
        ax.axvline(
            mean_val,
            color="black",
            linestyle="--",
            linewidth=1.3
        )

        # Gray dotted line = comparison group mean
        ax.axvline(
            ref_mean,
            color="gray",
            linestyle=":",
            linewidth=1.5
        )

        ax.set_title(family, fontsize=11)

        ax.text(
            0.98, 0.88,
            f"n = {n_obs:,} {obs_label}\n"
            f"mean = {mean_val:.2f}{mean_suffix}\n"
            f"{p_label}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8.5
        )

    for ax in axes[::5]:
        ax.set_ylabel(f"Number of {obs_label}")

    for ax in axes[5:]:
        ax.set_xlabel(xlabel)

    # Figure-level legend for reference lines
    legend_handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle="--",
            linewidth=1.5,
            label="Current panel mean"
        ),
        Line2D(
            [0], [0],
            color="gray",
            linestyle=":",
            linewidth=1.8,
            label="Comparison mean"
        )
    ]

    if show_legend:
        fig.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=2,
            frameon=False,
            fontsize=9
        )

    fig.suptitle(title, fontsize=14, y=1.08)
    plt.tight_layout()

    if save and filename is not None:
        fig.savefig(outdir / f"{filename}.pdf", bbox_inches="tight")

    plt.show()

In [ ]:
# ------------------------------------------------------------
# Plot 1: number of unique isoforms per gene
# ------------------------------------------------------------
n_isoform_bins = np.arange(0.5, gene_level_df["n_isoforms"].max() + 1.5, 1)
plot_family_metric_histograms(plot_df=gene_level_df,family_order=family_order,metric="n_isoforms",
    xlabel="Number of unique isoforms",title="Number of unique annotated isoforms per gene",
    filename="fig5A_tffamilies_nisoforms",bins=n_isoform_bins,obs_label="genes",mean_suffix=" iso/gene")

# ------------------------------------------------------------
# Plot 2: number of IDRs per isoform
# ------------------------------------------------------------
n_idr_bins = np.arange(-0.5, isoform_family_df["n_idr_segments"].max() + 1.5, 1)
plot_family_metric_histograms(plot_df=isoform_family_df,family_order=family_order,metric="n_idr_segments",
    xlabel="Number of IDRs",title="Number of IDRs per isoform",
    filename="fig5B_tf_families_nidrsegments",bins=n_idr_bins,obs_label="isoforms",mean_suffix=" IDRs/isoform")

# ------------------------------------------------------------
# Plot 3: percent disorder per isoform
# ------------------------------------------------------------
pct_idr_bins = np.linspace(0,100,21)
plot_family_metric_histograms(plot_df=isoform_family_df,family_order=family_order,metric="pct_idr",
    xlabel="% IDR",title="% disorder per isoform",
    filename="fig5C_tf_families_pctidr",bins=pct_idr_bins,obs_label="isoforms",mean_suffix="%")

# ------------------------------------------------------------
# Plot 4: isoform-level disorder variation per gene
# ------------------------------------------------------------
delta_idr_bins = np.linspace(0, 100, 21)

plot_family_metric_histograms(
    plot_df=gene_level_df,
    family_order=family_order,
    metric="pct_change_idr",
    xlabel="Isoform-level disorder variation (%IDR range)",
    title="Isoform-level %IDR variation per gene",
    filename="fig5D_tf_families_pctchangeidr",
    bins=delta_idr_bins,
    obs_label="genes",
    mean_suffix="% ΔIDR/gene"
)

## Save For Future Analysis

In [ ]:
# Removed: this cell overwrote data/annotated/IDRisoforms_df.csv and TranscriptionIsoforms_df.csv
# (the inputs read at the top of this notebook) with the age-complete subset built above
# (21,801 -> 21,653 rows), so re-running the notebook silently changed its own cohort.
# No figure or downstream table used those overwritten files.


# Further Plotting

In [ ]:
# ============================================================
# Canonical isoform representativeness
# Compare canonical %IDR with NONCANONICAL isoforms only
# ============================================================
all_isoforms_df = pd.read_csv(
    "../data/annotated/"
    "IDRisoforms_df_geneage.csv")

outdir = Path(
    "../figures/proteome")

outdir.mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------
# 2. Clean the isoform-level dataframe
# ------------------------------------------------------------

isoform_analysis_df = all_isoforms_df.copy()

required_columns = [
    "base_accession",
    "gene_name",
    "tf_group",
    "pct_idr",
    "pct_change_idr",
    "n_isoforms",
    "is_canonical"
]

missing_columns = [
    col for col in required_columns
    if col not in isoform_analysis_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# Convert pct_idr to numeric
isoform_analysis_df["pct_idr"] = pd.to_numeric(
    isoform_analysis_df["pct_idr"],
    errors="coerce"
)


# Convert is_canonical to Boolean if it was read as text
if not pd.api.types.is_bool_dtype(
    isoform_analysis_df["is_canonical"]
):
    isoform_analysis_df["is_canonical"] = (
        isoform_analysis_df["is_canonical"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )


isoform_analysis_df = (
    isoform_analysis_df
    .dropna(
        subset=[
            "base_accession",
            "tf_group",
            "pct_idr",
            "is_canonical"
        ]
    )
    .copy()
)

isoform_analysis_df["is_canonical"] = (
    isoform_analysis_df["is_canonical"].astype(bool)
)


# Remove duplicate isoform rows if an isoform identifier exists
if "isoform_accession" in isoform_analysis_df.columns:
    isoform_analysis_df = (
        isoform_analysis_df
        .drop_duplicates(
            subset=[
                "base_accession",
                "isoform_accession"
            ]
        )
        .copy()
    )


# ------------------------------------------------------------
# 3. Check canonical counts per gene
# ------------------------------------------------------------

canonical_counts = (
    isoform_analysis_df
    .groupby("base_accession", observed=True)
    .agg(
        n_isoforms_observed=("pct_idr", "size"),
        n_canonical=("is_canonical", "sum")
    )
    .reset_index()
)

print("Number of canonical rows per gene:")
display(
    canonical_counts["n_canonical"]
    .value_counts()
    .sort_index()
    .rename_axis("n_canonical")
    .reset_index(name="n_genes")
)


# Keep genes with:
# 1. Exactly one canonical isoform
# 2. At least three total isoforms
#    = one canonical + at least two noncanonical isoforms

valid_genes = (
    canonical_counts
    .query(
        "n_canonical == 1 and "
        "n_isoforms_observed > 2"
    )
    ["base_accession"]
)

isoform_analysis_df = (
    isoform_analysis_df[
        isoform_analysis_df["base_accession"].isin(
            valid_genes
        )
    ]
    .copy()
)

print(
    "Genes retained:",
    f"{isoform_analysis_df['base_accession'].nunique():,}"
)


# ------------------------------------------------------------
# 4. Calculate the mean of NONCANONICAL isoforms only
# ------------------------------------------------------------

noncanonical_summary_df = (
    isoform_analysis_df
    .loc[
        ~isoform_analysis_df["is_canonical"]
    ]
    .groupby(
        "base_accession",
        observed=True
    )
    .agg(
        mean_pct_idr_noncanonical=(
            "pct_idr",
            "mean"
        ),
        median_pct_idr_noncanonical=(
            "pct_idr",
            "median"
        ),
        min_pct_idr_noncanonical=(
            "pct_idr",
            "min"
        ),
        max_pct_idr_noncanonical=(
            "pct_idr",
            "max"
        ),
        sd_pct_idr_noncanonical=(
            "pct_idr",
            "std"
        ),
        n_noncanonical=(
            "pct_idr",
            "size"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Extract canonical isoform information
# ------------------------------------------------------------

canonical_df = (
    isoform_analysis_df
    .loc[
        isoform_analysis_df["is_canonical"],
        [
            "base_accession",
            "gene_name",
            "tf_group",
            "pct_idr",
            "pct_change_idr",
            "n_isoforms"
        ]
    ]
    .rename(
        columns={
            "pct_idr": "canonical_pct_idr"
        }
    )
    .copy()
)


# There should now be exactly one canonical row per gene
canonical_df = (
    canonical_df
    .drop_duplicates("base_accession")
    .copy()
)


# ------------------------------------------------------------
# 6. Merge canonical and noncanonical statistics
# ------------------------------------------------------------

canonical_representative_df = (
    canonical_df
    .merge(
        noncanonical_summary_df,
        on="base_accession",
        how="inner",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 7. Calculate signed and absolute deviation
# ------------------------------------------------------------

# Signed deviation:
# positive = canonical is more disordered
# negative = canonical is less disordered
canonical_representative_df[
    "canonical_signed_deviation_from_noncanonical"
] = (
    canonical_representative_df["canonical_pct_idr"]
    -
    canonical_representative_df[
        "mean_pct_idr_noncanonical"
    ]
)


# Absolute deviation:
# smaller = canonical is more representative
canonical_representative_df[
    "canonical_abs_deviation_from_noncanonical"
] = (
    canonical_representative_df[
        "canonical_signed_deviation_from_noncanonical"
    ]
    .abs()
)


# Optional direction labels
canonical_representative_df[
    "canonical_deviation_direction"
] = np.select(
    [
        canonical_representative_df[
            "canonical_signed_deviation_from_noncanonical"
        ] > 0,

        canonical_representative_df[
            "canonical_signed_deviation_from_noncanonical"
        ] < 0
    ],
    [
        "Canonical more disordered",
        "Canonical less disordered"
    ],
    default="Equal"
)


# Final cleaning
canonical_representative_df = (
    canonical_representative_df
    .dropna(
        subset=[
            "canonical_pct_idr",
            "pct_change_idr",
            "mean_pct_idr_noncanonical",
            "canonical_abs_deviation_from_noncanonical",
            "canonical_signed_deviation_from_noncanonical",
            "tf_group"
        ]
    )
    .copy()
)


print(
    "Final gene-level dataframe shape:",
    canonical_representative_df.shape
)

display(
    canonical_representative_df.head()
)

In [ ]:
# ============================================================
# Plot canonical representativeness
# ============================================================

def plot_canonical_representativeness(
    plot_df,
    save=True,
    filename=None
):
    plot_df = plot_df.copy()

    groups = ["Non-TF", "TF"]

    colors = {
        "Non-TF": "#4C72B0",
        "TF": "#DD8452"
    }

    # Wider figure to accommodate the third panel
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(19, 5.3)
    )

    # ========================================================
    # Panel A:
    # Canonical disorder versus total isoform-level variation
    # ========================================================

    ax = axes[0]

    for group in groups:

        sub = (
            plot_df
            .query("tf_group == @group")
            .dropna(
                subset=[
                    "canonical_pct_idr",
                    "pct_change_idr"
                ]
            )
        )

        ax.scatter(
            sub["canonical_pct_idr"],
            sub["pct_change_idr"],
            s=14,
            alpha=0.22,
            color=colors[group],
            edgecolors="none"
        )

        if sub.shape[0] > 2:
            x = sub["canonical_pct_idr"].to_numpy()
            y = sub["pct_change_idr"].to_numpy()

            m, b = np.polyfit(x, y, 1)

            x_line = np.linspace(
                x.min(),
                x.max(),
                100
            )

            ax.plot(
                x_line,
                m * x_line + b,
                color=colors[group],
                linewidth=2.2
            )


    rho, pval = spearmanr(
        plot_df["canonical_pct_idr"],
        plot_df["pct_change_idr"],
        nan_policy="omit"
    )

    ax.text(
        0.04,
        0.96,
        (
            "All genes\n"
            f"Spearman ρ = {rho:.3f}\n"
            f"p = {pval:.2e}"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            alpha=0.9
        )
    )

    ax.set_xlabel("Canonical isoform %IDR")
    ax.set_ylabel("Isoform-level Δ%IDR variation")

    ax.set_title(
        "A. Canonical disorder vs isoform variation",
        loc="left"
    )

    ax.set_xlim(-2, 102)
    ax.set_ylim(-2, 102)

    ax.grid(True, alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


    # ========================================================
    # Panel B:
    # Absolute deviation from NONCANONICAL isoform mean
    # ========================================================

    ax = axes[1]

    for group in groups:

        sub = (
            plot_df
            .query("tf_group == @group")
            .dropna(
                subset=[
                    "mean_pct_idr_noncanonical",
                    "canonical_abs_deviation_from_noncanonical"
                ]
            )
        )

        ax.scatter(
            sub["mean_pct_idr_noncanonical"],
            sub[
                "canonical_abs_deviation_from_noncanonical"
            ],
            s=14,
            alpha=0.22,
            color=colors[group],
            edgecolors="none"
        )

        if sub.shape[0] > 2:
            x = sub[
                "mean_pct_idr_noncanonical"
            ].to_numpy()

            y = sub[
                "canonical_abs_deviation_from_noncanonical"
            ].to_numpy()

            m, b = np.polyfit(x, y, 1)

            x_line = np.linspace(
                x.min(),
                x.max(),
                100
            )

            ax.plot(
                x_line,
                m * x_line + b,
                color=colors[group],
                linewidth=2.2
            )


    rho, pval = spearmanr(
        plot_df["mean_pct_idr_noncanonical"],
        plot_df[
            "canonical_abs_deviation_from_noncanonical"
        ],
        nan_policy="omit"
    )

    ax.text(
        0.04,
        0.96,
        (
            "All genes\n"
            f"Spearman ρ = {rho:.3f}\n"
            f"p = {pval:.2e}"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            alpha=0.9
        )
    )

    absolute_ylim = max(
        10,
        plot_df[
            "canonical_abs_deviation_from_noncanonical"
        ]
        .quantile(0.995)
        * 1.05
    )

    ax.set_xlabel(
        "Mean %IDR across noncanonical isoforms"
    )

    ax.set_ylabel(
        "|Canonical %IDR − mean noncanonical %IDR|"
    )

    ax.set_title(
        "B. Absolute deviation from alternatives",
        loc="left"
    )

    ax.set_xlim(-2, 102)
    ax.set_ylim(-2, absolute_ylim)

    ax.grid(True, alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


    # ========================================================
    # Panel C:
    # Signed deviation by TF group
    # ========================================================

    ax = axes[2]

    group_values = [
        (
            plot_df
            .query("tf_group == @group")
            ["canonical_signed_deviation_from_noncanonical"]
            .dropna()
            .to_numpy()
        )
        for group in groups
    ]


    boxplot = ax.boxplot(
        group_values,
        positions=[1, 2],
        widths=0.48,
        patch_artist=True,
        showfliers=False,
        medianprops={
            "color": "black",
            "linewidth": 2
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.2
        },
        capprops={
            "color": "black",
            "linewidth": 1.2
        },
        boxprops={
            "edgecolor": "black",
            "linewidth": 1.2
        }
    )


    for patch, group in zip(
        boxplot["boxes"],
        groups
    ):
        patch.set_facecolor(colors[group])
        patch.set_alpha(0.55)


    # Jittered points
    rng = np.random.default_rng(42)

    for position, group, values in zip(
        [1, 2],
        groups,
        group_values
    ):
        jitter = rng.normal(
            loc=position,
            scale=0.055,
            size=len(values)
        )

        ax.scatter(
            jitter,
            values,
            s=9,
            alpha=0.14,
            color=colors[group],
            edgecolors="none",
            rasterized=True
        )


        median_value = np.median(values)

        ax.text(
            position,
            0.97,
            (
                f"n = {len(values):,}\n"
                f"median = {median_value:.2f}"
            ),
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=9
        )


    # Zero indicates equal canonical and noncanonical mean
    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.5,
        alpha=0.8
    )


    # Compare TF and Non-TF signed deviations
    if all(len(values) > 0 for values in group_values):

        u_stat, p_group = mannwhitneyu(
            group_values[0],
            group_values[1],
            alternative="two-sided"
        )

        ax.text(
            0.04,
            0.04,
            (
                "TF vs Non-TF\n"
                f"Mann–Whitney p = {p_group:.2e}"
            ),
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox=dict(
                facecolor="white",
                edgecolor="black",
                alpha=0.9
            )
        )


    signed_limit = max(
        10,
        plot_df[
            "canonical_signed_deviation_from_noncanonical"
        ]
        .abs()
        .quantile(0.995)
        * 1.05
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(groups)

    ax.set_xlabel("Protein group")

    ax.set_ylabel(
        "Canonical %IDR − mean noncanonical %IDR"
    )

    ax.set_title(
        "C. Direction of canonical deviation",
        loc="left"
    )

    ax.set_ylim(
        -signed_limit,
        signed_limit
    )

    ax.grid(
        True,
        axis="y",
        alpha=0.18
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


    # ========================================================
    # Shared legend and title
    # ========================================================

    handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markersize=7,
            color=colors[group],
            label=(
                f"{group} "
                f"n={plot_df.query('tf_group == @group').shape[0]:,}"
            )
        )
        for group in groups
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.025)
    )

    fig.suptitle(
        (
            "Canonical isoform representativeness relative "
            "to noncanonical isoforms"
        ),
        fontsize=14,
        y=1.09
    )

    plt.tight_layout()


    if save and filename is not None:
        plt.savefig(
            outdir / f"{filename}.pdf",
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
plot_canonical_representativeness(
    canonical_representative_df,
    save=True,
    filename="fig6B_canonical_representativeness"
)

In [ ]:
# Canonical percentile within each gene's isoform %IDR range
# ------------------------------------------------------------

# Add min/max %IDR across isoforms for each gene
idr_range_df = (
    all_isoforms_df
    .groupby("base_accession")
    .agg(
        min_pct_idr=("pct_idr", "min"),
        max_pct_idr=("pct_idr", "max")
    )
    .reset_index()
)

canonical_representative_df = (
    canonical_representative_df
    .merge(idr_range_df, on="base_accession", how="left")
)

canonical_representative_df["canonical_percentile_in_idr_range"] = (
    canonical_representative_df["canonical_pct_idr"] - canonical_representative_df["min_pct_idr"]) / (
    canonical_representative_df["max_pct_idr"] - canonical_representative_df["min_pct_idr"])

# Avoid genes with no IDR range, where max == min
canonical_percentile_df = (
    canonical_representative_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["canonical_percentile_in_idr_range", "tf_group"])
    .copy()
)

canonical_percentile_df["canonical_percentile_in_idr_range"] = (
    canonical_percentile_df["canonical_percentile_in_idr_range"]
    .clip(0, 1)
)
print("hi")

In [ ]:
# Plot settings
groups = ["Non-TF", "TF"]

colors = {
    "Non-TF": "#4C72B0",
    "TF": "#DD8452"
}

bins = np.linspace(0, 1, 21)

fig, ax = plt.subplots(figsize=(8, 5))

for group in groups:
    vals = (
        canonical_percentile_df
        .query("tf_group == @group")
        ["canonical_percentile_in_idr_range"]
        .dropna()
    )

    # Prevent division by zero if a group has no genes
    if len(vals) == 0:
        print(f"No observations found for {group}")
        continue

    # Make the histogram show percent of genes within each group
    weights = np.full(len(vals), 100 / len(vals))

    ax.hist(
        vals,
        bins=bins,
        weights=weights,
        histtype="step",
        linewidth=2.2,
        color=colors[group],
        label=(
            f"{group} n={len(vals):,}, "
            f"median={vals.median():.2f}"
        )
    )

    ax.axvline(
        vals.median(),
        color=colors[group],
        linestyle="--",
        linewidth=2
    )

ax.axvline(
    0.5,
    color="black",
    linestyle=":",
    linewidth=2,
    label="Middle of min–max range"
)

ax.set_xlabel("Canonical position within isoform %IDR range")
ax.set_ylabel("Genes within group (%)")
ax.set_title(
    "Where does the canonical isoform fall within each gene's IDR range?"
)

ax.legend(frameon=False)
plt.tight_layout()

# Save before displaying
fig.savefig(
    outdir / "fig6A_canonical_position.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
canonical_representative_df.groupby("tf_group").agg(
    n_genes=("base_accession", "nunique"),
    mean_canonical_pct_idr=("canonical_pct_idr", "mean"),
    median_canonical_pct_idr=("canonical_pct_idr", "median"),
    mean_pct_change_idr=("pct_change_idr", "mean"),
    median_pct_change_idr=("pct_change_idr", "median"))

In [ ]:
# ============================================================
# Canonical length bias: canonical vs. alternative isoforms
# ============================================================

def _length_delta_panel(ax, df, color, label):
    """Compute and plot canonical-length minus alt-isoform-length for one group."""
    canonical_len = (
        df[df["is_canonical"]]
        .set_index("base_accession")["Length"]
        .rename("canonical_length")
    )
    alt_df = (
        df[~df["is_canonical"]]
        .join(canonical_len, on="base_accession")
    )
    alt_df["length_delta"] = alt_df["canonical_length"] - alt_df["Length"]
    deltas = alt_df["length_delta"].dropna()

    pct_pos = (deltas > 0).mean() * 100
    pct_neg = (deltas < 0).mean() * 100
    med     = deltas.median()

    p1, p99   = np.percentile(deltas, 1), np.percentile(deltas, 99)
    n_clipped = ((deltas < p1) | (deltas > p99)).sum()

    ax.axvspan(p1, 0,   alpha=0.07, color="#E05C5C", zorder=0)
    ax.axvspan(0,  p99, alpha=0.07, color=color,     zorder=0)

    ax.hist(deltas.clip(p1, p99), bins=60, color=color, edgecolor="none", alpha=0.80)
    ax.axvline(0,   color="#333333", linewidth=1.5, linestyle="--", zorder=3, label=r"$\Delta$ = 0")
    ax.axvline(med, color="#E05C5C", linewidth=1.5, linestyle="--", zorder=3,
               label=f"median = +{med:.0f} aa")

    n_genes = df["base_accession"].nunique()
    ax.set_title(
        f"{label}\n(N = {len(deltas):,} alt isoforms, {n_genes:,} genes)",
        fontsize=12
    )
    ax.set_xlabel("Canonical length − Isoform length (aa)", fontsize=11)
    ax.set_ylabel("Number of isoforms", fontsize=11)

    ax.text(
        0.97, 0.95,
        f"Canonical longer (Δ > 0): {pct_pos:.1f}%\nIsoform longer  (Δ < 0): {pct_neg:.1f}%",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9)
    )
    if n_clipped > 0:
        ax.text(0.02, 0.98, f"{n_clipped} outliers clipped (p1–p99)",
                transform=ax.transAxes, ha="left", va="top", fontsize=8, color="gray")
    ax.legend(frameon=False, fontsize=10)


fig, (ax_tf, ax_nontf) = plt.subplots(1, 2, figsize=(14, 5))

_length_delta_panel(ax_tf,    tf_isoforms_df,                                       "#DD8452", "TF isoforms")
_length_delta_panel(ax_nontf, all_isoforms_df[all_isoforms_df["tf_group"] == "Non-TF"], "#4C72B0", "Non-TF isoforms")

plt.suptitle("Are canonical isoforms longer than their alternatives?", fontsize=13, y=1.01)
plt.tight_layout()

fig.savefig(outdir / "fig6C_canonical_length_delta.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Same figure — no clipping (full x-axis range)

def _length_delta_panel_unclipped(ax, df, color, label):
    canonical_len = (
        df[df["is_canonical"]]
        .set_index("base_accession")["Length"]
        .rename("canonical_length")
    )
    alt_df = (
        df[~df["is_canonical"]]
        .join(canonical_len, on="base_accession")
    )
    alt_df["length_delta"] = alt_df["canonical_length"] - alt_df["Length"]
    deltas = alt_df["length_delta"].dropna()

    med = deltas.median()

    ax.axvspan(deltas.min(), 0,            alpha=0.07, color="#E05C5C", zorder=0)
    ax.axvspan(0,            deltas.max(), alpha=0.07, color=color,     zorder=0)

    ax.hist(deltas, bins=60, color=color, edgecolor="none", alpha=0.80)
    ax.axvline(0,   color="#333333", linewidth=1.5, linestyle="--", zorder=3, label=r"$\Delta$ = 0")
    ax.axvline(med, color="#E05C5C", linewidth=1.5, linestyle="--", zorder=3,
               label=f"median = +{med:.0f} aa")

    ax.set_title(f"{label} (unclipped)", fontsize=12)
    ax.set_xlabel("Canonical length − Isoform length (aa)", fontsize=11)
    ax.set_ylabel("Number of isoforms", fontsize=11)
    ax.legend(frameon=False, fontsize=10)


fig, (ax_tf, ax_nontf) = plt.subplots(1, 2, figsize=(14, 5))

_length_delta_panel_unclipped(ax_tf,    tf_isoforms_df,                                        "#DD8452", "TF isoforms")
_length_delta_panel_unclipped(ax_nontf, all_isoforms_df[all_isoforms_df["tf_group"] == "Non-TF"], "#4C72B0", "Non-TF isoforms")

plt.suptitle("Are canonical isoforms longer than their alternatives? (full range)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Number of IDR segments: TF vs Non-TF, Canonical vs Alternative
# ============================================================

# Build a combined group label for plotting
plot_df = all_isoforms_df.copy()
plot_df["iso_type"] = plot_df["is_canonical"].map({True: "Canonical", False: "Alternative"})

# Ordered categories and colors
order    = ["TF", "Non-TF"]
hue_order = ["Canonical", "Alternative"]
palette  = {
    "Canonical":   {"TF": "#DD8452", "Non-TF": "#4C72B0"},
    "Alternative": {"TF": "#F5B88A", "Non-TF": "#95B8D9"},
}
flat_palette = {
    ("TF",     "Canonical"):   "#DD8452",
    ("TF",     "Alternative"): "#F5B88A",
    ("Non-TF", "Canonical"):   "#4C72B0",
    ("Non-TF", "Alternative"): "#95B8D9",
}

fig, ax = plt.subplots(figsize=(9, 5))

sns.violinplot(
    data=plot_df,
    x="tf_group", y="n_idr_segments",
    hue="iso_type",
    order=order, hue_order=hue_order,
    palette={"Canonical": "#DD8452", "Alternative": "#F5B88A"},
    inner=None, linewidth=0.8, alpha=0.75, ax=ax,
    cut=0,
)

# Re-color Non-TF violins (seaborn applies hue palette uniformly; patch the artists)
violin_artists = [c for c in ax.collections if hasattr(c, "get_paths")]
group_labels   = [(g, h) for g in order for h in hue_order]
for artist, (grp, hue) in zip(violin_artists, group_labels):
    if grp == "Non-TF":
        artist.set_facecolor({"Canonical": "#4C72B0", "Alternative": "#95B8D9"}[hue])

# Overlay median dots and % zero annotation per group
x_positions = {("TF", "Canonical"): -0.20, ("TF", "Alternative"): 0.20,
               ("Non-TF", "Canonical"): 0.80, ("Non-TF", "Alternative"): 1.20}

for (grp, hue), x in x_positions.items():
    vals = plot_df[(plot_df["tf_group"] == grp) & (plot_df["iso_type"] == hue)]["n_idr_segments"].dropna()
    med  = vals.median()
    pct_zero = (vals == 0).mean() * 100
    color = flat_palette[(grp, hue)]

    ax.scatter(x, med, color="white", edgecolor=color, s=60, zorder=5, linewidth=1.5)
    ax.text(x, -1.8, f"{pct_zero:.0f}%\nno IDR", ha="center", va="top",
            fontsize=8, color=color, fontweight="bold")

# Mann-Whitney U: canonical vs alternative within each TF group
for i, grp in enumerate(order):
    can = plot_df[(plot_df["tf_group"] == grp) & (plot_df["iso_type"] == "Canonical")]["n_idr_segments"].dropna()
    alt = plot_df[(plot_df["tf_group"] == grp) & (plot_df["iso_type"] == "Alternative")]["n_idr_segments"].dropna()
    stat, p = mannwhitneyu(can, alt, alternative="greater")
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    ax.text(i, plot_df["n_idr_segments"].max() * 0.92, f"canon > alt\n{sig} (p={p:.2e})",
            ha="center", fontsize=8.5, color="black",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#cccccc", alpha=0.85))

ax.set_xlabel("")
ax.set_ylabel("Number of IDR segments per isoform", fontsize=12)
ax.set_title("IDR segment count: TF vs Non-TF, Canonical vs Alternative", fontsize=13)
ax.set_ylim(-3, None)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#DD8452", label="TF Canonical"),
    Patch(facecolor="#F5B88A", label="TF Alternative"),
    Patch(facecolor="#4C72B0", label="Non-TF Canonical"),
    Patch(facecolor="#95B8D9", label="Non-TF Alternative"),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10, loc="upper right")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Transcription Factors", "Non-TF Proteins"], fontsize=11)

plt.tight_layout()
fig.savefig(outdir / "fig7A_n_idr_segments_tf_canonical.pdf", bbox_inches="tight")
plt.show()